# Day 6 — Dimensionality Reduction
**Week 3 · Feature Engineering & Data Prep · AI Engineer Lab**

Reduce high-dimensional data while preserving structure. This notebook covers:
- PCA: theory, scree plots, explained variance, 2D visualisation
- PCA as preprocessing for downstream models
- Interpreting principal components (loading vectors)
- t-SNE: cluster visualisation (visualisation only)
- UMAP: faster, global-structure-preserving alternative
- TruncatedSVD for sparse matrices (NLP use case)
- When to use which technique

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_digits, fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

np.random.seed(42)
print('Libraries loaded.')

## 1. Load the Digits Dataset — 64 Pixel Features

In [ ]:
digits = load_digits()
X = digits.data       # (1797, 64) — each sample is an 8x8 greyscale image flattened
y = digits.target     # 0-9 digit labels

print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features (8x8 pixels)')
print(f'Classes: {np.unique(y)}')

# Visualise a few digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for digit in range(10):
    ax = axes[digit // 5, digit % 5]
    idx = np.where(y == digit)[0][0]
    ax.imshow(X[idx].reshape(8, 8), cmap='gray_r')
    ax.set_title(f'Digit: {digit}', fontsize=10)
    ax.axis('off')
plt.suptitle('Digits Dataset Samples', fontsize=12)
plt.tight_layout()
plt.show()

## 2. PCA — Finding Principal Components

In [ ]:
# Always standardise before PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit full PCA to study explained variance
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)

# Find k for 90%, 95%, 99% variance
for threshold in [0.80, 0.90, 0.95, 0.99]:
    k = np.argmax(cumvar >= threshold) + 1
    print(f'{threshold*100:.0f}% variance explained by {k} components (of 64 original)')

# Scree plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, 21), pca_full.explained_variance_ratio_[:20] * 100,
            color='#3b82f6', alpha=0.8)
axes[0].set_title('Explained Variance per Component (first 20)')
axes[0].set_xlabel('Component')
axes[0].set_ylabel('Explained variance (%)')

axes[1].plot(range(1, 65), cumvar * 100, color='#22c55e', linewidth=2)
for thresh in [0.80, 0.90, 0.95]:
    k = np.argmax(cumvar >= thresh) + 1
    axes[1].axhline(y=thresh*100, linestyle='--', alpha=0.6,
                     label=f'{thresh*100:.0f}%: {k} components')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative variance (%)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. PCA 2D Visualisation — Seeing the Classes

In [ ]:
pca_2d = PCA(n_components=2, random_state=42)
X_2d = pca_2d.fit_transform(X_scaled)

print(f'Variance explained by 2 PCs: {pca_2d.explained_variance_ratio_.sum()*100:.1f}%')

plt.figure(figsize=(10, 7))
colors = plt.cm.tab10(np.linspace(0, 1, 10))
for digit in range(10):
    mask = y == digit
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1],
                c=[colors[digit]], label=str(digit),
                alpha=0.6, s=20)

plt.title('Digits Dataset — PCA 2D Projection', fontsize=13)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend(title='Digit', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

## 4. Interpreting Principal Components

In [ ]:
# pca.components_ shows the loading vectors — how much each pixel contributes to each PC
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

pca_vis = PCA(n_components=10, random_state=42)
pca_vis.fit(X_scaled)

for i, ax in enumerate(axes.flatten()):
    component = pca_vis.components_[i].reshape(8, 8)
    ax.imshow(component, cmap='RdBu_r', vmin=-0.3, vmax=0.3)
    var_pct = pca_vis.explained_variance_ratio_[i] * 100
    ax.set_title(f'PC{i+1} ({var_pct:.1f}%)', fontsize=9)
    ax.axis('off')

plt.suptitle('First 10 Principal Components (Loading Vectors)\nRed=positive, Blue=negative loading', fontsize=11)
plt.tight_layout()
plt.show()
print('PC1 captures the overall intensity pattern (bright vs dark overall).')
print('Later PCs capture finer structural differences between digit shapes.')

## 5. PCA as Preprocessing — Model Performance

In [ ]:
lr = LogisticRegression(max_iter=2000, random_state=42)

experiments = {
    'All 64 features': Pipeline([('sc', StandardScaler()), ('lr', lr)]),
    'PCA 20 components': Pipeline([('sc', StandardScaler()), ('pca', PCA(n_components=20)), ('lr', lr)]),
    'PCA 95% variance': Pipeline([('sc', StandardScaler()), ('pca', PCA(n_components=0.95)), ('lr', lr)]),
    'PCA 2 components (vis)': Pipeline([('sc', StandardScaler()), ('pca', PCA(n_components=2)), ('lr', lr)]),
}

print('=== PCA Preprocessing Impact (5-Fold Accuracy) ===')
for name, pipe in experiments.items():
    cv = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
    # Get number of components if PCA is in pipeline
    print(f'  {name:<30}  Accuracy = {cv.mean():.4f} ± {cv.std():.4f}')

## 6. t-SNE — Cluster Visualisation

In [ ]:
# t-SNE is slow — use a subsample or PCA first to speed it up
# Common approach: PCA to 50D, then t-SNE to 2D
pca_50 = PCA(n_components=50, random_state=42)
X_pca50 = pca_50.fit_transform(X_scaled)

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000, learning_rate='auto')
X_tsne = tsne.fit_transform(X_pca50)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PCA 2D
for digit in range(10):
    mask = y == digit
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], c=[colors[digit]],
                    label=str(digit), alpha=0.5, s=15)
axes[0].set_title('PCA (2D)\nSome class overlap', fontsize=11)
axes[0].legend(title='Digit', bbox_to_anchor=(1.05, 1))

# t-SNE 2D
for digit in range(10):
    mask = y == digit
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=[colors[digit]],
                    label=str(digit), alpha=0.5, s=15)
axes[1].set_title('t-SNE (2D)\nClear cluster separation', fontsize=11)
axes[1].legend(title='Digit', bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.show()
print('t-SNE shows much clearer clusters. BUT: axes are meaningless, global distances are distorted.')
print('t-SNE is ONLY for visualisation — never use its components as model features.')

## 7. TruncatedSVD for Sparse Text Data (LSA)

In [ ]:
# NLP use case: reduce TF-IDF matrix to dense semantic vectors
# Load 4 newsgroups categories
categories = ['sci.med', 'sci.space', 'rec.sport.hockey', 'rec.autos']
newsgroups = fetch_20newsgroups(subset='train', categories=categories,
                                 remove=('headers', 'footers', 'quotes'),
                                 random_state=42)

# TF-IDF: sparse matrix
tfidf = TfidfVectorizer(max_features=5000, min_df=5, stop_words='english')
X_tfidf = tfidf.fit_transform(newsgroups.data)
print(f'TF-IDF matrix: {X_tfidf.shape}, sparsity: {1 - X_tfidf.nnz / np.prod(X_tfidf.shape):.2%}')

# TruncatedSVD (cannot centre sparse matrices like PCA does)
svd = TruncatedSVD(n_components=100, random_state=42)
X_svd = svd.fit_transform(X_tfidf)
print(f'After TruncatedSVD: {X_svd.shape}')
print(f'Variance explained: {svd.explained_variance_ratio_.sum()*100:.1f}%')

# Compare logistic regression on TF-IDF vs TruncatedSVD
lr_pipe_tfidf = Pipeline([('lr', LogisticRegression(max_iter=1000))])
lr_pipe_svd   = Pipeline([('lr', LogisticRegression(max_iter=1000))])

cv_tfidf = cross_val_score(lr_pipe_tfidf, X_tfidf, newsgroups.target, cv=3, scoring='accuracy').mean()
cv_svd   = cross_val_score(lr_pipe_svd, X_svd, newsgroups.target, cv=3, scoring='accuracy').mean()

print(f'\nLogistic Regression on TF-IDF (5000 features): {cv_tfidf:.4f}')
print(f'Logistic Regression on SVD-100 (100 features):  {cv_svd:.4f}')
print('SVD dramatically reduces dimensionality with minimal accuracy loss!')

## 8. Reconstruction Error — How Much Information Did We Lose?

In [ ]:
# PCA reconstruction: project to k components, then reconstruct
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
n_components_list = [2, 10, 30, 64]

sample_idx = [np.where(y == d)[0][0] for d in range(8)]
X_samples = X_scaled[sample_idx]

for row, k in enumerate(n_components_list):
    pca_k = PCA(n_components=k, random_state=42)
    X_compressed = pca_k.fit_transform(X_samples)
    X_reconstructed = pca_k.inverse_transform(X_compressed)

    for col in range(8):
        ax = axes[row, col]
        ax.imshow(X_reconstructed[col].reshape(8, 8), cmap='gray_r')
        ax.axis('off')
        if col == 0:
            var_pct = sum(pca_k.explained_variance_ratio_) * 100
            ax.set_ylabel(f'{k} PCs\n({var_pct:.0f}%)', fontsize=8, rotation=0, labelpad=50)

plt.suptitle('PCA Reconstruction at Different Component Counts', fontsize=12)
plt.tight_layout()
plt.show()

## Exercises

1. **Wine dataset PCA**: Apply PCA to the Wine dataset (13 features). How many components explain 99% variance? Visualise the 3 wine classes in 2D PCA space.

2. **t-SNE perplexity sweep**: Run t-SNE with perplexity=[5, 15, 30, 50, 100] on the digits dataset. How does cluster separation change? What is the "right" perplexity?

3. **PCA reconstruction RMSE**: For the digits dataset, compute reconstruction RMSE for k=2,5,10,20,30,40,50,64 components. Plot RMSE vs k. At what k does reconstruction become visually acceptable?

4. **Topic words in SVD**: After TruncatedSVD on newsgroups, inspect `svd.components_` to find the top 10 words with highest loading in each of the first 5 components. What topic does each component represent?

In [ ]:
# ── Exercise 4 Solution: SVD topic words ──
feature_names_tfidf = tfidf.get_feature_names_out()

print('=== Top 10 Words per SVD Component (Topics) ===')
for i in range(5):
    top_words_idx = svd.components_[i].argsort()[-10:][::-1]
    top_words = [feature_names_tfidf[j] for j in top_words_idx]
    print(f'Component {i+1}: {top_words}')